In [ ]:
import os
from pathlib import Path

BRANCH = os.environ.get("TINYFORGE_BRANCH", "t4-kaggle-d12")
REPO = os.environ.get("TINYFORGE_REPO", "https://github.com/michaelp91-dev/tinyforge.git")


%cd /content
!rm -rf /content/tinyforge
!git clone -b {BRANCH} {REPO}
%cd /content/tinyforge

!nvidia-smi -L
!free -g
!python -m pip install -q torch numpy tokenizers transformers datasets pyarrow requests tiktoken gguf safetensors
print("cwd", Path(".").resolve())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/llm.c/models"
os.makedirs(DRIVE, exist_ok=True)
print("Drive exports go to", DRIVE)

In [ ]:
!make train_tinyforge GPU=t4
!ls -l train_tinyforge

In [ ]:
import os
os.makedirs("/content/data", exist_ok=True)
!python python/train_tokenizer.py --out /content/data/tokenizer --vocab-size 32768 --max-chars 50000000
!python python/prepare_climbmix.py --tokenizer /content/data/tokenizer --out /content/data/climbmix --target-tokens 30000000 --val-tokens 262144
!ls -lh /content/data/climbmix
!free -g

In [ ]:
%%bash
rm -rf /content/smoke
./train_tinyforge \
  -i "/content/data/climbmix/train_*.bin" -j /content/data/climbmix/val.bin \
  -e d12 -o /content/smoke \
  -b 16 -t 512 -d 65536 -x 60 \
  -v 30 -m 10 -n 0 -y 1 --keep-ckpts 2 --time-limit 420 \
  --vocab 32768 \
  --token-bytes /content/data/tokenizer/token_bytes.bin 2>&1 | tee /content/smoke1.log

In [ ]:
%%bash
./train_tinyforge \
  -i "/content/data/climbmix/train_*.bin" -j /content/data/climbmix/val.bin \
  -e d12 -o /content/smoke \
  -b 16 -t 512 -d 65536 -x 60 \
  -v 30 -m 10 -n 0 -y 1 --keep-ckpts 2 \
  --vocab 32768 \
  --token-bytes /content/data/tokenizer/token_bytes.bin 2>&1 | tee /content/smoke2.log

In [ ]:
import re, statistics, shutil
from pathlib import Path

pat = re.compile(r"step (\d+)/(\d+)\s+loss (\S+)\s+lr_mult \S+\s+(\S+)s\s+(\d+) tok/s")
rows = []
text = ""
for name in ["smoke1.log", "smoke2.log"]:
    t = Path("/content", name).read_text()
    text += t
    rows += [(int(m[1]), float(m[3]), float(m[4]), int(m[5])) for m in pat.finditer(t)]
for line in text.splitlines():
    if line.startswith(("device:", "parameters:", "GPU memory", "resumed", "time limit", "  val")):
        print(line)
first, last = rows[0], rows[-1]
speeds = [r[3] for r in rows[3:]] or [r[3] for r in rows]
tok_s = statistics.median(speeds)
print(f"steps run: {len(rows)}   loss {first[1]:.3f} -> {last[1]:.3f}")
print("LOSS FELL" if last[1] < first[1] - 2.0 else "LOSS DID NOT FALL ENOUGH - stop and check")
print(f"one T4: {tok_s:,.0f} tok/s")
kaggle = tok_s * 1.9
for hours in (20, 25):
    steps = int(kaggle * hours * 3600 / 262144)
    print(f"Kaggle 2xT4 ~{kaggle:,.0f} tok/s -> {hours} h = {steps} steps ({steps * 262144 / 1e6:,.0f}M tokens)")
report = Path(DRIVE) / "tinyforge_t4_smoke.log"
report.write_text(text)
print("log saved to", report)

In [ ]:
import os, shutil
from pathlib import Path

dones = sorted(Path("/content/smoke").glob("DONE_*"))
step = int(dones[-1].name.split("_")[1])
ckpt = f"/content/smoke/model_{step:08d}.bin"
!python python/export_hf.py --input {ckpt} --output /content/smoke-hf --tokenizer /content/data/tokenizer
!python python/convert_tinyforge_to_gguf.py --model /content/smoke-hf --out /content/smoke-f32.gguf
!ls -lh /content/smoke-hf /content/smoke-f32.gguf